# Solar Active Region Cutouts
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edoardolegnaro/KTH_ML4Space2026/blob/main/Solar_AR_cutouts.ipynb)

**KTH PhD Course — ARCCnet Practice Session**

In this notebook we use a toy version of the **ARCCnet Active Region cutout dataset** (https://www.aanda.org/articles/aa/full_html/2026/06/aa58722-25/aa58722-25.html).  
Each entry contains a pair of FITS images (magnetogram + continuum) of solar active regions (AR), along with all metadata.

The notebook is organised in two parts:

1. **Exploratory Data Analysis (EDA)** — quality filtering, class distributions, temporal and spatial coverage, pixel statistics, and image properties.
2. **Modelling** — pre-processing, train/validation/test splitting, and a progression of baseline classifiers (MLP → CNN → 2D multimodal fusion CNN).

### Dataset structure
```
arccnet-ar-classification-toy-v20251016/
├── region_classification.parq   ← metadata (3 000 rows)
└── fits/                        ← FITS cutout pairs
    ├── *_mag_MDI.fits            ← magnetogram (SOHO/MDI  1996–2010)
    ├── *_cont_MDI.fits           ← continuum   (SOHO/MDI  1996–2010)
    ├── *_mag_HMI.fits            ← magnetogram (SDO/HMI   2010–2022)
    └── *_cont_HMI.fits           ← continuum   (SDO/HMI   2010–2022)
```

## Setup

Run the cell below once to install any missing packages.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

_REQUIRED = {
    # Core EDA
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "astropy": "astropy",
    "pyarrow": "pyarrow",
    # ML
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "torch": "torch",
    "tqdm": "tqdm",
    "p_tqdm": "p_tqdm",
    "tensorboard": "tensorboard",
    "gdown": "gdown",
}

_missing = [_pkg for _module, _pkg in _REQUIRED.items()
            if importlib.util.find_spec(_module) is None]

if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *_missing
    ])
else:
    print("All required packages already available.")

# In Colab, imports may need cache invalidation after pip installs.
importlib.invalidate_caches()

import os
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from tqdm import tqdm
from p_tqdm import p_map

# Make the local teaching utilities importable from either repo root or notebook dir.
_REPO_NAME = "KTH_ML4Space2026"
_GITHUB_RAW_BASE = f"https://raw.githubusercontent.com/edoardolegnaro/{_REPO_NAME}/main"

if IN_COLAB and not (Path.cwd() / "utils.py").exists():
    import urllib.request

    print("Downloading utils.py from GitHub ...")
    urllib.request.urlretrieve(f"{_GITHUB_RAW_BASE}/utils.py", "utils.py")

for _utils_dir in [Path.cwd(), Path.cwd() / _REPO_NAME, Path.cwd() / "KTH_PhD_Course", Path("/content") / _REPO_NAME, Path("/content") / "KTH_PhD_Course"]:
    if _utils_dir.exists() and str(_utils_dir) not in sys.path:
        sys.path.insert(0, str(_utils_dir))

import utils as ut

print("Running in Google Colab:" if IN_COLAB else "Running locally:", IN_COLAB)
print("All imports loaded.")

### EDA configuration

Edit this cell for dataset discovery and exploratory-analysis thresholds.


In [ ]:
# -- EDA configuration --------------------------------------------------------
# Leave as None for automatic discovery, or set to an explicit path, e.g.
# DATASET_DIR = Path("/content/drive/MyDrive/arccnet-ar-classification-toy-v20251016")
DATASET_DIR = None

# Optional: paste a public Google Drive URL to a zipped dataset folder.
# Expected archive contents: region_classification.parq and fits/*.fits, either
# directly or inside arccnet-ar-classification-toy-v20251016/.
DATASET_GOOGLE_DRIVE_URL = "https://drive.google.com/file/d/1lvzSZblxuszFTdGSucQhKatBXnfQMryi/view?usp=sharing"

# Colab only: set True if the dataset is already stored in your Google Drive.
MOUNT_GOOGLE_DRIVE = False

# EDA filters and display knobs.
LON_LIMIT_DEG = 75          # projected-radius limb exclusion threshold
NAN_THRESHOLD = 0.05        # remove images with more than this NaN fraction
PREPROCESS_N_SHOW = 4       # number of AR examples in the preprocessing preview
PREPROCESS_RANDOM_STATE = 7

# Labels, class grouping, and plot colours.
MDI_COLOR = "royalblue"
HMI_COLOR = "tomato"
AR5_CLASSES = ["Alpha", "Beta", "Beta-Gamma", "Beta-Delta", "Beta-Gamma-Delta"]
LABEL_COLORS = {
    "QS": "#5E9BD4",
    "IA": "#77C785",
    "Alpha": "#E9C46A",
    "Beta": "#2A9D8F",
    "Beta-Gamma": "#F4A261",
    "Beta-Delta": "#E76F51",
    "Beta-Gamma-Delta": "#264653",
    "Gamma": "#A06AB4",
    "Gamma-Delta": "#6A3D7A",
}
QS_IA_AR_MAPPING = {
    "QS": "QS",
    "IA": "IA",
    "Alpha": "AR",
    "Beta": "AR",
    "Beta-Delta": "AR",
    "Beta-Gamma": "AR",
    "Beta-Gamma-Delta": "AR",
    "Gamma": "AR",
    "Gamma-Delta": "AR",
}
COARSE_COLORS = {"QS": "#5E9BD4", "IA": "#77C785", "AR": "#EF6C5C"}
AR5_CLASS_COLORS = {label: LABEL_COLORS[label] for label in AR5_CLASSES}

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if DATASET_GOOGLE_DRIVE_URL and DATASET_DIR is None:
    DATASET_DIR = ut.download_google_drive_dataset(
        DATASET_GOOGLE_DRIVE_URL,
        dest_dir=Path("/content") if IN_COLAB else Path.cwd(),
        dataset_name=ut.DATASET_NAME,
    )

DATASET_DIR = ut.find_dataset_dir(DATASET_DIR, dataset_name=ut.DATASET_NAME)
PARQUET_FILE = DATASET_DIR / "region_classification.parq"
FITS_DIR = DATASET_DIR / "fits"

print(f"Dataset: {DATASET_DIR}")
print(f"Parquet: {PARQUET_FILE}")
print(f"FITS files: {len(list(FITS_DIR.glob('*.fits'))):,}")
print(f"Limb threshold: {LON_LIMIT_DEG} deg")
print(f"NaN threshold: {NAN_THRESHOLD:.0%}")


In [ ]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)


---
## Exploratory Data Analysis

### Load the dataset

In [ ]:
# Load metadata and derive date/year/label/instrument columns.
df_raw = ut.load_dataset(PARQUET_FILE)
print(f"Rows: {len(df_raw):,}    Columns: {df_raw.shape[1]}")

df_raw

### Quality filtering

The dataset includes a handful of rows flagged as bad-quality.  
We decode the hex quality flags and remove them before deeper analysis.

In [ ]:
# Quality flag helpers live in utils.py; here we inspect the two instruments.
print("=== SDO/HMI quality flags ===")
display(ut.summarise_quality(df_raw, "QUALITY_hmi", "HMI"))

print("=== SOHO/MDI quality flags ===")
display(ut.summarise_quality(df_raw, "QUALITY_mdi", "MDI"))


In [ ]:
# Keep only rows with at least one valid, good-quality cutout path.
df = ut.clean_dataset(df_raw)

print(f"Before filtering : {len(df_raw):,} rows")
print(f"After  filtering : {len(df):,} rows  ({len(df_raw) - len(df)} removed)")


### Class distribution

The label hierarchy is:

```
QS  — Quiet Sun
IA  — Plage Regions
AR  — Active Region  (Alpha / Beta / Beta-Gamma / Beta-Delta / Beta-Gamma-Delta / Gamma / Gamma-Delta)
```

In [ ]:
ut.plot_class_histogram(df["label"], title="Clean Dataset Class Distribution",
                     color_map=LABEL_COLORS)

In [ ]:
# Coarser 3-class view: QS / IA / AR using the same mapping as the ARCCnet project
df["label_coarse"] = df["label"].map(QS_IA_AR_MAPPING)

ut.plot_class_histogram(
    df["label_coarse"],
    title="QS / IA / AR",
    color_map=COARSE_COLORS,
    figsize=(7, 5),
)


In [ ]:
# Break down by instrument as well
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, inst, color in zip(axes, ["HMI", "MDI"], [HMI_COLOR, MDI_COLOR]):
    sub  = df[df["instrument"] == inst]["label"]
    cnt  = sub.value_counts()
    clrs = [LABEL_COLORS.get(l, "#888888") for l in cnt.index]
    ax.bar(cnt.index, cnt.values, color=clrs, edgecolor="k", linewidth=0.5)
    ax.set_title(f"{inst} ({len(sub):,} samples)", fontsize=13)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
    for i, (lbl, val) in enumerate(cnt.items()):
        ax.text(i, val + 5, str(val), ha="center", fontsize=9)
plt.suptitle("Label distribution per instrument", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### Temporal coverage

The dataset spans two solar instruments:
- **SOHO/MDI** — Solar & Heliospheric Observatory, Michelson Doppler Imager (1996–2010)
- **SDO/HMI**  — Solar Dynamics Observatory, Helioseismic & Magnetic Imager (2010–2022)

In [ ]:
# ── year histograms per instrument ──────────────────────────────────────────

mdi_df = df[df["instrument"] == "MDI"]
hmi_df = df[df["instrument"] == "HMI"]

mdi_counts = mdi_df["dates"].dt.to_period("M").value_counts().sort_index()
hmi_counts = hmi_df["dates"].dt.to_period("M").value_counts().sort_index()

mdi_idx = mdi_counts.index.to_timestamp()
hmi_idx = hmi_counts.index.to_timestamp()

tick_dates = [datetime(y, 1, 1) for y in range(1996, 2024, 2)]

with plt.style.context("seaborn-v0_8-darkgrid"):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8),
                                   sharex=True,
                                   gridspec_kw={"height_ratios": [5, 1]})

    ax1.bar(mdi_idx, mdi_counts.values, width=20, color=MDI_COLOR, alpha=0.85, label="MDI")
    ax1.bar(hmi_idx, hmi_counts.values, width=20, color=HMI_COLOR, alpha=0.85, label="HMI")
    ax1.set_ylabel("Samples / month", fontsize=15)
    ax1.legend(fontsize=14)
    ax1.set_title("Temporal distribution of samples", fontsize=18)
    ax1.tick_params(axis="y", labelsize=12)
    ax1.grid(True, linestyle="--", alpha=0.5)

    ax2.vlines(mdi_df["dates"].values, 0.1, 0.9, color=MDI_COLOR, alpha=0.4, linewidth=0.3)
    ax2.vlines(hmi_df["dates"].values, 1.1, 1.9, color=HMI_COLOR, alpha=0.4, linewidth=0.3)
    ax2.set(ylim=[0, 2], yticks=[], xlabel="")
    ax2.xaxis_date()
    ax2.xaxis.set_major_locator(mdates.YearLocator(2))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax2.set_xticks(tick_dates)
    plt.setp(ax2.get_xticklabels(), rotation=45, fontsize=13)
    ax2.set_xlabel("Year", fontsize=15)

    plt.tight_layout()
    plt.show()

print(f"MDI span : {mdi_df['dates'].min().date()} → {mdi_df['dates'].max().date()}  ({len(mdi_df):,} samples)")
print(f"HMI span : {hmi_df['dates'].min().date()} → {hmi_df['dates'].max().date()}  ({len(hmi_df):,} samples)")


### Spatial distribution on the solar disk

Active regions tend to appear in the **activity belt** (roughly ±35° latitude from the equator), migrating towards the equator over each ~11-year solar cycle.

In [ ]:
# Select coordinate columns (HMI for HMI rows, MDI for MDI rows)
lat = np.where(
    df["instrument"] == "HMI",
    pd.to_numeric(df["latitude_hmi"],  errors="coerce"),
    pd.to_numeric(df["latitude_mdi"],  errors="coerce"),
)
lon = np.where(
    df["instrument"] == "HMI",
    pd.to_numeric(df["longitude_hmi"], errors="coerce"),
    pd.to_numeric(df["longitude_mdi"], errors="coerce"),
)

# Keep only AR rows for the spatial plots (QS / IA have no AR coordinates)
ar_mask = df["label_coarse"] == "AR"
lat_ar = lat[ar_mask]
lon_ar = lon[ar_mask]
valid  = np.isfinite(lat_ar) & np.isfinite(lon_ar)
lat_ar, lon_ar = lat_ar[valid], lon_ar[valid]

# ── Latitude / Longitude histograms ─────────────────────────────────────────
degree_ticks = np.arange(-90, 91, 15)

with plt.style.context("seaborn-v0_8-darkgrid"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.hist(lon_ar, bins=np.arange(-90, 91, 1), color="steelblue", edgecolor="k", linewidth=0.3)
    ax1.set_xticks(degree_ticks)
    ax1.set_xticklabels([f"{d}°" for d in degree_ticks], fontsize=8)
    ax1.set_xlabel("Longitude (°)", fontsize=12)
    ax1.set_ylabel("Count", fontsize=12)
    ax1.set_title("Heliographic longitude of ARs", fontsize=13)

    ax2.hist(lat_ar, bins=np.arange(-60, 61, 1), color="tomato", edgecolor="k", linewidth=0.3)
    ax2.set_xticks(np.arange(-60, 61, 15))
    ax2.set_xticklabels([f"{d}°" for d in np.arange(-60, 61, 15)], fontsize=8)
    ax2.set_xlabel("Latitude (°)", fontsize=12)
    ax2.set_title("Heliographic latitude of ARs", fontsize=13)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Solar-disk projection ────────────────────────────────────────────────────
# Convert heliographic coordinates to disk (y, z) plane
#   y = cos(lat) * sin(lon)
#   z = sin(lat)

lat_r = np.deg2rad(lat_ar)
lon_r = np.deg2rad(lon_ar)
yV    = np.cos(lat_r) * np.sin(lon_r)
zV    = np.sin(lat_r)

fig, ax = plt.subplots(figsize=(8, 8))
ax.add_artist(plt.Circle((0, 0), 1, edgecolor="gray", facecolor="#f8f4ef", linewidth=1))
ut.solar_grid(ax)
ax.scatter(yV, zV, s=5, alpha=0.35, color="steelblue", rasterized=True)
ax.set(xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), aspect="equal")
ax.axis("off")
ax.set_title("Active regions projected onto the solar disk", fontsize=13, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Butterfly diagram: latitude vs. time for AR rows ────────────────────────
ar_rows   = df[ar_mask].copy()
ar_rows["lat"] = lat[ar_mask]
ar_rows["lon"] = lon[ar_mask]
ar_rows   = ar_rows.dropna(subset=["lat", "lon"])

with plt.style.context("seaborn-v0_8-darkgrid"):
    fig, ax = plt.subplots(figsize=(16, 5))

    ax.scatter(
        ar_rows["dates"], ar_rows["lat"],
        c="steelblue", s=8, alpha=0.5, rasterized=True
    )

    ax.axhline(0, color="k", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_ylabel("Heliographic latitude (°)", fontsize=12)
    ax.set_xlabel("Year", fontsize=12)
    ax.set_title("Butterfly diagram", fontsize=14)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.get_xticklabels(), rotation=30, fontsize=10)
    plt.tight_layout()
    plt.show()


#### Reading the butterfly diagram

The **butterfly diagram** (also called the **Spörer diagram**) plots the heliographic latitude of each active region against time.

- At the **start of each ~11-year solar cycle**, active regions emerge at mid-latitudes (~±30–35°).
- As the cycle matures they drift systematically **toward the equator**, reaching ~±5–10° at solar minimum.
- The next cycle then restarts at mid-latitudes, producing the characteristic **wing** shape on each side of the equator.

| Period | Solar cycle | Notes |
|---|---|---|
| 1996 – 2008 | Cycle 23 | SOHO/MDI data; clear equator-ward drift |
| 2008 – 2009 | Minimum | Very few ARs, near-equatorial |
| 2010 – 2019 | Cycle 24 | SDO/HMI data; notably quieter cycle |
| 2019 – 2022 | Cycle 25 | SDO/HMI data; new cycle, ARs reappearing at mid-latitudes |

The diagram also serves as a **sanity check**: points near the poles or randomly scattered would indicate coordinate errors in the dataset.

### Longitude filtering (limb exclusion)

Active regions near the solar limb suffer from **projection effects** that make their magnetograms unreliable. The standard mitigation is to exclude regions close to the limb. Here we use a **projected-radius** threshold (`lon_limit_deg=65`).

In [ ]:
# -- Illustrate the limb projection problem -----------------------------------
# Pick one AR near the disk centre and one close to the limb,
# then show their cutouts side-by-side with disk-position insets.

_ar_limb_vis = ar_rows.copy()
_ar_limb_vis["_r"] = np.sqrt(
    (np.cos(np.deg2rad(_ar_limb_vis["lat"])) * np.sin(np.deg2rad(_ar_limb_vis["lon"]))) ** 2
    + np.sin(np.deg2rad(_ar_limb_vis["lat"])) ** 2
)

# Centre example: r < 0.2
_centre_candidates = _ar_limb_vis[_ar_limb_vis["_r"] < 0.2].dropna(subset=["lat", "lon"])
# Limb example: take the most extreme longitude available (highest r), sorted descending
_limb_candidates = (
    _ar_limb_vis[_ar_limb_vis["_r"] > 0.92]
    .dropna(subset=["lat", "lon"])
    .sort_values("_r", ascending=False)
)

if _centre_candidates.empty or _limb_candidates.empty:
    print("Not enough candidates for this illustration in the toy dataset.")
else:
    _centre_row = _centre_candidates.sample(1, random_state=3).iloc[0]
    # Pick from the top-10 most extreme to allow some variety, but bias toward the limb
    _limb_row   = _limb_candidates.head(10).sample(1, random_state=4).iloc[0]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle("Projection effect: disk centre vs. solar limb", fontsize=14)

    for row_i, (ex_row, label) in enumerate([
        (_centre_row, f"Disk centre  (lon = {_centre_row['lon']:.1f}°, lat = {_centre_row['lat']:.1f}°)"),
        (_limb_row,   f"Near limb    (lon = {_limb_row['lon']:.1f}°,  lat = {_limb_row['lat']:.1f}°)"),
    ]):
        try:
            mag, cont = ut.load_fits_pair(ex_row, FITS_DIR)
        except Exception as e:
            print(f"Could not load example: {e}")
            continue

        ms, cs = ut.pixel_stats(mag), ut.pixel_stats(cont)
        vlim = max(abs(ms["p1"]), abs(ms["p99"]))

        # Magnetogram
        im1 = axes[row_i, 0].imshow(mag, cmap="RdBu_r", origin="lower", vmin=-vlim, vmax=vlim)
        axes[row_i, 0].set_title(f"{label}\nMagnetogram  {mag.shape[0]}×{mag.shape[1]} px", fontsize=9)
        axes[row_i, 0].axis("off")
        plt.colorbar(im1, ax=axes[row_i, 0], shrink=0.8, label="G")

        # Continuum
        im2 = axes[row_i, 1].imshow(cont, cmap="gray", origin="lower", vmin=cs["p1"], vmax=cs["p99"])
        axes[row_i, 1].set_title(f"{label}\nContinuum  {cont.shape[0]}×{cont.shape[1]} px", fontsize=9)
        axes[row_i, 1].axis("off")
        plt.colorbar(im2, ax=axes[row_i, 1], shrink=0.8)

        # Solar-disk inset showing position
        ax_disk = axes[row_i, 2]
        _lat_r = np.deg2rad(ex_row["lat"])
        _lon_r = np.deg2rad(ex_row["lon"])
        _y = np.cos(_lat_r) * np.sin(_lon_r)
        _z = np.sin(_lat_r)
        _r = np.sqrt(_y**2 + _z**2)

        ax_disk.add_artist(plt.Circle((0, 0), 1, ec="gray", fc="#f8f4ef", lw=1))
        ut.solar_grid(ax_disk)
        ax_disk.scatter([_y], [_z], s=200, color="#E76F51", zorder=5, marker="*",
                        label=f"r = {_r:.2f}")
        ax_disk.set(xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), aspect="equal")
        ax_disk.axis("off")
        ax_disk.legend(fontsize=10, loc="lower right")
        ax_disk.set_title("Position on solar disk", fontsize=9)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Longitude filter for limb exclusion (projected-radius version) ──────────
# Projected radius on the solar disk
lat_r_lb = np.deg2rad(lat_ar)
lon_r_lb = np.deg2rad(lon_ar)
yV_lb    = np.cos(lat_r_lb) * np.sin(lon_r_lb)
zV_lb    = np.sin(lat_r_lb)
r        = np.sqrt(yV_lb**2 + zV_lb**2)
r_limb   = np.sin(np.deg2rad(LON_LIMIT_DEG))

# Retain points whose projected radius is within the limb boundary
keep = r <= r_limb

n_total = len(r)
n_keep  = keep.sum()
n_rm    = (~keep).sum()

print(f"AR rows with valid coordinates : {n_total:,}")
print(f"Retained  r ≤ sin({LON_LIMIT_DEG}°) : {n_keep:,}  ({n_keep / n_total * 100:.1f}%)")
print(f"Removed   r > sin({LON_LIMIT_DEG}°) : {n_rm:,}  ({n_rm / n_total * 100:.1f}%)")

# ── Visualise ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))

ax.add_artist(plt.Circle((0, 0), 1, ec="gray", fc="#f8f4ef", lw=1))
ut.solar_grid(ax)

# Exclusion-zone boundary
ax.add_artist(plt.Circle((0, 0), r_limb, ec="#E76F51", fc="none",
                          lw=2, ls="--"))

# Excluded points (dark) underneath retained points (bright)
ax.scatter(yV_lb[~keep], zV_lb[~keep], s=6, alpha=0.55,
           color="#4a4a4a", label=f"Excluded   r > sin({LON_LIMIT_DEG}°)",
           rasterized=True)
ax.scatter(yV_lb[keep],  zV_lb[keep],  s=6, alpha=0.6,
           color="#00BFFF", label=f"Retained   r ≤ sin({LON_LIMIT_DEG}°)",
           rasterized=True)

ax.set(xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), aspect="equal")
ax.axis("off")
ax.legend(fontsize=10, loc="lower right")
ax.set_title(f"Limb filtering — r ≤ sin({LON_LIMIT_DEG}°)", fontsize=13, pad=10)
plt.tight_layout()
plt.show()

### Physical properties

The parquet contains sunspot-group metadata: **area** (in millionths of a solar hemisphere, MSH), **number of sunspots**, and **longitudinal extent**.

In [ ]:
phys_cols = ["area", "number_of_sunspots", "longitudinal_extent"]
phys_df   = ar_rows[["label"] + phys_cols].copy()
for c in phys_cols:
    phys_df[c] = pd.to_numeric(phys_df[c], errors="coerce")

# Keep a consistent 5-class AR subset so boxplots share the same palette
phys_df_5 = phys_df[phys_df["label"].isin(AR5_CLASSES)].copy()
ar5_order = [l for l in AR5_CLASSES if l in phys_df_5["label"].unique()]

#display(phys_df_5.groupby("label")[phys_cols].describe().T)

titles = ["Area (MSH)", "Number of sunspots", "Longitudinal extent (°)"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col, title in zip(axes, phys_cols, titles):
    sub = phys_df_5.dropna(subset=[col])
    sns.boxplot(
        data=sub, x="label", y=col, order=ar5_order,
        palette=AR5_CLASS_COLORS, ax=ax,
        flierprops=dict(marker=".", markersize=2, alpha=0.3),
    )
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=40, labelsize=9)

plt.suptitle("Physical properties by magnetic class", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### Sample FITS images

Each entry provides two co-registered cutouts:  
- **Magnetogram** — line-of-sight magnetic field (bright = positive polarity, dark = negative)  
- **Continuum** — visible-light intensity image (shows sunspots as dark umbra/penumbra)

In [ ]:
# -- Display one example per magnetic class -----------------------------------
classes_to_show = [label for label in LABEL_COLORS if label in df["label"].unique()]

for target_label in classes_to_show:
    candidates = df[df["label"] == target_label]
    if candidates.empty:
        continue

    row = candidates.sample(1, random_state=42).iloc[0]
    mag, cont = ut.load_fits_pair(row, FITS_DIR)
    ms, cs = ut.pixel_stats(mag), ut.pixel_stats(cont)
    inst = row.get("instrument", "?")
    date = str(row.get("dates", ""))[:10]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f"{target_label}  |  {inst}  |  {date}", fontsize=12, y=0.92)

    # Magnetogram: symmetric clipping for visual clarity.
    vlim = max(abs(ms["p1"]), abs(ms["p99"]))
    im1 = ax1.imshow(mag, cmap="RdBu_r", origin="lower", vmin=-vlim, vmax=vlim)
    ax1.set_title(f"Magnetogram\nmean={ms['mean']:.1f}  std={ms['std']:.1f}", fontsize=10)
    ax1.axis("off")
    plt.colorbar(im1, ax=ax1, shrink=0.75, label="Gauss")

    im2 = ax2.imshow(cont, cmap="gray", origin="lower", vmin=cs["p1"], vmax=cs["p99"])
    ax2.set_title(f"Continuum\nmean={cs['mean']:.0f}  std={cs['std']:.0f}", fontsize=10)
    ax2.axis("off")
    plt.colorbar(im2, ax=ax2, shrink=0.75)

    plt.tight_layout()
    plt.show()


### Pixel-value statistics

In [ ]:
# -- Compute stats for all AR rows --------------------------------------------
def _compute_pixel_stats(idx):
    """Load FITS pair, compute pixel stats, return a record dict or None."""
    try:
        row = df.iloc[idx]
        mag, cont = ut.load_fits_pair(row, FITS_DIR)
        ms, cs = ut.pixel_stats(mag), ut.pixel_stats(cont)
        record = {"df_index": idx, "label": row["label"]}
        for key, value in ms.items():
            record[f"mag_{key}"] = value
        for key, value in cs.items():
            record[f"cont_{key}"] = value
        return record
    except Exception:
        return None


_raw = p_map(_compute_pixel_stats, range(len(df)), desc="Pixel stats")
_good_mask = np.array([r is not None for r in _raw])
stats_df = pd.DataFrame([r for r in _raw if r is not None])
print(f"Loaded stats for {len(stats_df)} images (failed to load: {(~_good_mask).sum()}).")

# ── NaN fraction per class ────────────────────────────────────────────────────
# Most images have zero NaN pixels. Show: (left) count of affected images per
# class; (right) NaN fraction only for those affected images.
label_order_all = [l for l in LABEL_COLORS if l in stats_df["label"].unique()]
palette_all = {l: LABEL_COLORS.get(l, "#888") for l in label_order_all}

nan_counts = (
    stats_df[stats_df["mag_nan_fraction"] > 0]
    .groupby("label")
    .size()
    .reindex(label_order_all, fill_value=0)
)
nan_affected = stats_df[stats_df["mag_nan_fraction"] > 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Left: how many images are affected per class
colors_ = [palette_all[l] for l in label_order_all]
bars = ax1.bar(label_order_all, nan_counts.values, color=colors_, edgecolor="k", linewidth=0.5)
for bar, val in zip(bars, nan_counts.values):
    if val > 0:
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 str(val), ha="center", va="bottom", fontsize=9)
ax1.set_title("Images with any NaN pixels", fontsize=12)
ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=35)
ax1.set_ylim(0, max(nan_counts.values) * 1.1)

# Right: strip plot of the non-zero NaN fractions
if not nan_affected.empty:
    sns.stripplot(
        data=nan_affected, x="label", y="mag_nan_fraction",
        order=[l for l in label_order_all if l in nan_affected["label"].values],
        palette=palette_all, ax=ax2, size=5, jitter=True, alpha=0.7,
    )
ax2.set_title("NaN fraction (affected images only)", fontsize=12)
ax2.set_xlabel("")
ax2.set_ylabel("NaN fraction")
ax2.tick_params(axis="x", rotation=35)

plt.suptitle("Magnetogram NaN pixels by class", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── Overall pixel-value distribution (histograms) ───────────────────────────
stat_cols = [
    ("mag_mean",  "Magnetogram Mean (G)",   "royalblue"),
    ("mag_std",   "Magnetogram Std (G)",    "royalblue"),
    ("cont_mean", "Continuum Mean (DN)",    "tomato"),
    ("cont_std",  "Continuum Std (DN)",     "tomato"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (col, title, color) in zip(axes.ravel(), stat_cols):
    ax.hist(stats_df[col].dropna(), bins=40, color=color, alpha=0.7,
            edgecolor="k", linewidth=0.4)
    ax.set_yscale("log")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count (log)")
    ax.grid(True, alpha=0.3)
plt.suptitle("Distribution of image statistics (AR sample)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Inspect the stats row with the highest mean magnetogram value.
_max_idx = stats_df["mag_mean"].idxmax()
stats_df.loc[_max_idx]


In [ ]:
# ── Inspect the magnetogram with the highest mean ────────────────────────────
# An unusually high |mag_mean| may indicate an artefact or corrupted FITS file.

_max_idx = stats_df["mag_mean"].idxmax()
_max_row = stats_df.loc[_max_idx]

row = df.iloc[int(_max_row["df_index"])]
mag, cont = ut.load_fits_pair(row, FITS_DIR)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle(
    f"Outlier magnetogram  |  mag_mean = {_max_row['mag_mean']:.0f} G  |  "
    f"label = {_max_row['label']}  |  {row.get('instrument', '?')}  |  "
    f"{str(row.get('dates', ''))[:10]}",
    fontsize=12, y=0.98,
)

vlim = max(abs(np.nanpercentile(mag, 1)), abs(np.nanpercentile(mag, 99)))
im1 = ax1.imshow(mag, cmap="RdBu_r", origin="lower", vmin=-vlim, vmax=vlim)
ax1.set_title(f"Magnetogram\nmean={_max_row['mag_mean']:.0f}  std={_max_row['mag_std']:.0f}", fontsize=10)
ax1.axis("off")
plt.colorbar(im1, ax=ax1, shrink=0.75, label="G")

cs = ut.pixel_stats(cont)
im2 = ax2.imshow(cont, cmap="gray", origin="lower", vmin=cs["p1"], vmax=cs["p99"])
ax2.set_title(f"Continuum\nmean={cs['mean']:.0f}  std={cs['std']:.0f}", fontsize=10)
ax2.axis("off")
plt.colorbar(im2, ax=ax2, shrink=0.75)

# Histogram of magnetogram pixel values
finite_mag = mag[np.isfinite(mag)]
ax3.hist(finite_mag.ravel(), bins=64, color="royalblue", alpha=0.7, edgecolor="k", linewidth=0.3)
ax3.axvline(0, color="k", lw=0.8, ls="--")
ax3.set_title(f"Pixel-value histogram  (n={len(finite_mag):,})", fontsize=10)
ax3.set_xlabel("Magnetic field (G)")
ax3.set_ylabel("Count")
ax3.set_yscale("log")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Filter out images with too many NaN pixels from the dataset ──────────────
_mask = stats_df["mag_nan_fraction"] <= NAN_THRESHOLD
_n_removed = (~_mask).sum()

_keep_idx = stats_df.loc[_mask, "df_index"].astype(int).to_numpy()
df = df.iloc[_keep_idx].reset_index(drop=True)
stats_df = stats_df.loc[_mask].reset_index(drop=True)
stats_df["df_index"] = np.arange(len(stats_df))

if _n_removed:
    print(f"Removed {_n_removed} image{'s' if _n_removed > 1 else ''} with > {NAN_THRESHOLD:.0%} NaN pixels.")
else:
    print(f"No images with > {NAN_THRESHOLD:.0%} NaN pixels found — no rows removed.")
print(f"Dataset size after NaN filtering: {len(df)} rows")


In [ ]:
# ── Boxplots of mag_mean and cont_mean per magnetic class ────────────────────
label_order_ar = [l for l in LABEL_COLORS if l in stats_df["label"].unique()]
palette_ar     = {l: LABEL_COLORS.get(l, "#888") for l in label_order_ar}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=stats_df, x="label", y="mag_mean",
            order=label_order_ar, palette=palette_ar, ax=ax1,
            flierprops=dict(marker=".", markersize=3, alpha=0.4))
ax1.set_title("Magnetogram mean by class", fontsize=13)
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=40)
ax1.axhline(0, color="k", lw=0.8, ls="--")

sns.boxplot(data=stats_df, x="label", y="cont_mean",
            order=label_order_ar, palette=palette_ar, ax=ax2,
            flierprops=dict(marker=".", markersize=3, alpha=0.4))
ax2.set_title("Continuum mean by class", fontsize=13)
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=40)

plt.tight_layout()
plt.show()


### Image dimensions

Cutout sizes vary because the physical extent of active regions differs.  
Most ML models require a fixed input size — explore how images are distributed.

In [ ]:
# -- Collect image shapes from `dim_image_cutout_*` columns --------------------
dim_records = []
for _, row in df.iterrows():
    col = "dim_image_cutout_hmi" if row["instrument"] == "HMI" else "dim_image_cutout_mdi"
    h, w = ut.parse_dim(row.get(col))
    if h is not None:
        dim_records.append({"label": row["label"], "instrument": row["instrument"],
                             "height": h, "width": w})

dim_df = pd.DataFrame(dim_records)

if dim_df.empty or "height" not in dim_df.columns:
    print("No parseable dimension data found in 'dim_image_cutout_*' columns.")
    print("Falling back to reading shapes directly from a sample of FITS files ...")

    _sample = df.sample(min(200, len(df)), random_state=0).reset_index(drop=True)

    def _read_shape(idx):
        try:
            row = _sample.iloc[idx]
            mag, _ = ut.load_fits_pair(row, FITS_DIR)
            return {"label": row["label"], "instrument": row["instrument"],
                    "height": mag.shape[0], "width": mag.shape[1]}
        except Exception:
            return None

    shape_records = [
        r for r in p_map(_read_shape, range(len(_sample)), desc="Reading FITS shapes")
        if r is not None
    ]
    dim_df = pd.DataFrame(shape_records)

if not dim_df.empty and "height" in dim_df.columns:
    print(dim_df[["height", "width"]].describe().to_string())
else:
    print("Could not determine image dimensions.")


In [ ]:
if not dim_df.empty:
    fig = plt.figure(figsize=(8, 6))
    gs  = fig.add_gridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                           hspace=0.08, wspace=0.08)

    ax_scatter = fig.add_subplot(gs[1, 0])
    ax_hist_x  = fig.add_subplot(gs[0, 0], sharex=ax_scatter)
    ax_hist_y  = fig.add_subplot(gs[1, 1], sharey=ax_scatter)
    fig.add_subplot(gs[0, 1]).set_visible(False)  # corner

    for inst, color in [("HMI", HMI_COLOR), ("MDI", MDI_COLOR)]:
        sub = dim_df[dim_df["instrument"] == inst]
        ax_scatter.scatter(sub["width"], sub["height"], c=color, alpha=0.5, s=12, label=inst)
        ax_hist_x.hist(sub["width"],  bins=30, alpha=0.6, color=color, edgecolor="k", linewidth=0.3)
        ax_hist_y.hist(sub["height"], bins=30, alpha=0.6, color=color, edgecolor="k", linewidth=0.3,
                       orientation="horizontal")

    ax_scatter.set_xlabel("Width (px)", fontsize=12)
    ax_scatter.set_ylabel("Height (px)", fontsize=12)
    ax_scatter.legend(fontsize=11)
    ax_hist_x.tick_params(labelbottom=False)
    ax_hist_x.set_yscale("log")
    ax_hist_y.tick_params(labelleft=False)
    ax_hist_y.set_xscale("log")
    fig.suptitle("Cutout image dimensions — scatter + marginal histograms", fontsize=14, y=0.98)
    plt.show()
else:
    print("No dimension data available in this version of the parquet.")


#### Why are cutout sizes variable?

The cutout algorithm crops a fixed **angular radius** around the active-region centre.  
The resulting pixel size on the detector depends on three factors:

* **Limb foreshortening** — regions near the solar limb appear compressed in longitude due to the viewing angle (projection effect), producing smaller cutouts.
* **Instrument plate scale** — MDI ∼0.5″/px vs HMI ∼0.6″/px, so the same angular extent maps to slightly different pixel counts.
* **AR geometry** — the bounding rectangle is fitted to each region's shape; elongated or compact ARs naturally give different aspect ratios.

As a result, the dataset contains everything from tiny near-limb snippets to large disk-centred cutouts. The pre-processing step will later pad and resize all images to a common shape.

---

## Summary

| Item | Finding |
|---|---|
| **Instruments** | SOHO/MDI (1996–2010) and SDO/HMI (2010–2022) |
| **Activity belt** | ARs concentrate within ±35° latitude |
| **Butterfly diagram** | Visible equator-ward drift over each solar cycle |
| **Pixel values** | Complex ARs tend to show stronger magnetic flux (higher \|mean\| and std) |
| **Image sizes** | Variable; padding/resizing needed before model training |

---

---
## Modelling

### Configuration

In [ ]:
# -- Model training configuration ---------------------------------------------
# All downstream image preprocessing uses TARGET_SIZE x TARGET_SIZE pixels.
TARGET_SIZE = 64

# Group-aware split configuration. With 10 folds, the defaults give 60/20/20.
SPLIT_N_FOLDS = 10
SPLIT_RANDOM_STATE = 42
TEST_FOLDS = [0, 1]
VAL_FOLDS = [2, 3]
TRAIN_FOLDS = [4, 5, 6, 7, 8, 9]

# Data loading and training behavior.
LOAD_IMAGES_IN_MEMORY = True
DATALOADER_NUM_WORKERS = 0
SHOW_PROGRESS_BARS = True
# Early stopping can monitor "val_loss", "val_acc", or "val_macro_f1".
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_METRIC = "val_macro_f1"
EARLY_STOPPING_MIN_DELTA = 1e-3
RESTORE_BEST_WEIGHTS = True

# Optimisation settings.
N_EPOCHS = 30
LR = 1e-3
MLP_EPOCHS = 30
MLP_LR = 1e-3

# TensorBoard logging. In Colab or Jupyter, view runs with:
# ut.show_tensorboard(TB_LOG_DIR)
ENABLE_TENSORBOARD = True
TB_LOG_DIR = "runs"

assert isinstance(TARGET_SIZE, int) and TARGET_SIZE > 0, "TARGET_SIZE must be a positive integer."
assert sorted(TEST_FOLDS + VAL_FOLDS + TRAIN_FOLDS) == list(range(SPLIT_N_FOLDS)), "Split folds must cover each fold exactly once."
print(f"Configured cutout size: {TARGET_SIZE} x {TARGET_SIZE} pixels")
print(f"Split folds: train={TRAIN_FOLDS}, val={VAL_FOLDS}, test={TEST_FOLDS}")
print(f"Early stopping: {EARLY_STOPPING_METRIC}, patience={EARLY_STOPPING_PATIENCE}, min_delta={EARLY_STOPPING_MIN_DELTA}")
print(f"TensorBoard logging: {'on' if ENABLE_TENSORBOARD else 'off'}")


### Pre-processing

Before feeding images into a model, each cutout must be:

1. **Cleaned** — replace NaN / ±Inf pixels with **0** (zero field).
2. **Physically normalised** — divide by a fixed **800 G** scale factor, then clamp to **[−1, 1]** (hardtanh).  
   Positive pixels → positive polarity; negative pixels → negative polarity.  
3. **Padded** — zero-pad to a square canvas (preserving aspect ratio); 0 is the neutral midpoint in [−1, 1].
4. **Resized** — scale to the configured fixed spatial size (`TARGET_SIZE × TARGET_SIZE`).


In [ ]:
# -- Pre-processing function and visualisation --------------------------------

fig, axes = plt.subplots(PREPROCESS_N_SHOW, 2, figsize=(7, 2.5 * PREPROCESS_N_SHOW))
loaded = 0

for _, row in df[df["label_coarse"] == "AR"].sample(PREPROCESS_N_SHOW * 3, random_state=PREPROCESS_RANDOM_STATE).iterrows():
    if loaded >= PREPROCESS_N_SHOW:
        break
    try:
        mag, _ = ut.load_fits_pair(row, FITS_DIR)
    except Exception:
        continue

    proc = ut.preprocess_image(mag, target_size=TARGET_SIZE)
    finite = mag[np.isfinite(mag)]
    vlim = max(abs(np.percentile(finite, 1)), abs(np.percentile(finite, 99)))

    axes[loaded, 0].imshow(mag, cmap="RdBu_r", origin="lower", vmin=-vlim, vmax=vlim)
    axes[loaded, 0].set_title(f"Raw  {mag.shape[0]}x{mag.shape[1]}", fontsize=9)
    axes[loaded, 0].axis("off")

    axes[loaded, 1].imshow(proc, cmap="RdBu_r", origin="lower", vmin=-1, vmax=1)
    axes[loaded, 1].set_title(f"Pre-processed  {TARGET_SIZE}x{TARGET_SIZE}  [-1, 1]", fontsize=9)
    axes[loaded, 1].axis("off")

    loaded += 1

plt.suptitle("Pre-processing pipeline", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


### Train / Validation / Test split

We work with the **coarse 3-class problem** (QS / IA / AR).

Key consideration: the **same active region** can appear on multiple consecutive days (as it rotates across the disk).  
A random split would leak information — the same AR might be in both train and test.

We prevent this using **`StratifiedGroupKFold`**:
- Rows are grouped by the tracked-region `number` column for **all** classes (QS / IA / AR).
- This keeps repeated observations of the same numbered region in only one split.
- The fold assignment respects group boundaries, so no group straddles train, validation, and test.
- 10-fold split → **fold 0 = test (10%), folds 1–2 = val (20%), folds 3–9 = train (70%)**.

In [ ]:
# -- Train / validation / test indices ----------------------------------------
# Coarse 3-class problem; magnetogram only (simplest baseline input).
ml_sample = df.copy().reset_index(drop=True)
labels = ml_sample["label_coarse"].values

# Group-aware splitting: same region number cannot appear in multiple splits.
region_col = next(
    col for col in ["number", "noaa_number", "noaa_ar", "ar_number", "harpnum", "nar"]
    if col in ml_sample.columns
)

region_number = pd.to_numeric(ml_sample[region_col], errors="coerce")
if region_number.isna().any():
    n_missing = region_number.isna().sum()
    display(ml_sample.loc[region_number.isna(), ["dates", "label", "label_coarse"]].head())
    raise ValueError(f"{n_missing} rows have no parseable {region_col}; cannot group by number.")

groups = ("region_" + region_number.astype(int).astype(str)).values
print(f"Grouping strategy: {region_col} for all QS / IA / AR rows.")

# StratifiedGroupKFold split defined in the model configuration cell.
le = LabelEncoder()
y_enc = le.fit_transform(labels)
sgkf = StratifiedGroupKFold(n_splits=SPLIT_N_FOLDS, shuffle=True, random_state=SPLIT_RANDOM_STATE)
folds = list(sgkf.split(np.zeros(len(y_enc)), y_enc, groups=groups))

test_idx = np.concatenate([folds[i][1] for i in TEST_FOLDS])
val_idx = np.concatenate([folds[i][1] for i in VAL_FOLDS])
train_idx = np.concatenate([folds[i][1] for i in TRAIN_FOLDS])

print(f"{'Split':<8} {'N':>5}   " + "  ".join(f"{c:>5}" for c in le.classes_))
print("-" * (20 + 7 * len(le.classes_)))
for name, idx in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
    counts = dict(zip(*np.unique(labels[idx], return_counts=True)))
    row_str = "  ".join(f"{counts.get(c, 0):>5}" for c in le.classes_)
    print(f"{name:<8} {len(idx):>5}   {row_str}")

# Verify no region-number leakage between splits.
for a_name, a_idx, b_name, b_idx in [
    ("train", train_idx, "val", val_idx),
    ("train", train_idx, "test", test_idx),
    ("val", val_idx, "test", test_idx),
]:
    a_numbers = set(region_number.iloc[a_idx].astype(int))
    b_numbers = set(region_number.iloc[b_idx].astype(int))
    assert not (a_numbers & b_numbers), f"Region-number leakage between {a_name} and {b_name}."

print("Region-number leakage check passed for all classes.")

# Numeric labels used by the model sections.
y = y_enc

MODEL_RESULTS = {}

print(f"Neural-network loading strategy: {'in-memory TensorDataset' if LOAD_IMAGES_IN_MEMORY else 'on-the-fly ut.SolarFITSDataset'}")
print(f"Progress bars during training: {SHOW_PROGRESS_BARS}")
print("ut.SolarFITSDataset ready.")

In [ ]:
# ── Visualise train / val / test class distributions ────────────────────────

split_names = ["Train", "Validation", "Test"]
split_indices = [train_idx, val_idx, test_idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, sname, sidx in zip(axes, split_names, split_indices):
    sub_labels = labels[sidx]
    counts = pd.Series(sub_labels).value_counts()
    total = counts.sum()

    # Fixed order: QS, AR, IA
    order = [c for c in ["QS", "AR", "IA"] if c in counts.index]
    vals = [counts[c] for c in order]
    colors_ = [COARSE_COLORS.get(c, "#888888") for c in order]

    bars = ax.bar(order, vals, color=colors_, edgecolor="k", linewidth=0.5)

    for bar, val in zip(bars, vals):
        pct = val / total * 100
        label = f"{val:,} ({pct:.1f}%)"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + total * 0.01,
                label, ha="center", va="bottom", fontsize=10)

    ax.set_title(f"{sname}  ({total:,} samples)", fontsize=14)
    ax.set_ylabel("Count", fontsize=12)
    ax.tick_params(axis="both", labelsize=11)
    # Add vertical headroom so percentage labels are not clipped
    ymax = max(vals) * 1.08
    ax.set_ylim(0, ymax)

plt.suptitle("Class distribution per split  (QS / IA / AR)", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()



In [ ]:
# -- Preload magnetograms and configure training ------------------------------
DEVICE, DEVICE_BACKEND = ut.select_torch_device()

class_counts = np.bincount(y)
class_weight_np = len(y) / (len(le.classes_) * class_counts)
class_weight = torch.tensor(class_weight_np, dtype=torch.float32, device=DEVICE)

TB_COMMON_HPARAMS = {
    "target_size": TARGET_SIZE,
    "n_train": len(train_idx),
    "n_val": len(val_idx),
    "n_test": len(test_idx),
    "device_backend": DEVICE_BACKEND,
    "load_images_in_memory": LOAD_IMAGES_IN_MEMORY,
}

print(f"Training device: {DEVICE_BACKEND} ({DEVICE})")
print("Class weights:", dict(zip(le.classes_, class_weight_np.round(3))))
if ENABLE_TENSORBOARD:
    print(f"TensorBoard logging enabled in {TB_LOG_DIR!r}; view with ut.show_tensorboard(TB_LOG_DIR).")


def _load_mag(idx, _target_size=TARGET_SIZE):
    """Load and preprocess one magnetogram as a 2-D float32 image."""
    row = ml_sample.iloc[idx]
    mag, _ = ut.load_fits_pair(row, FITS_DIR)
    return ut.preprocess_image(mag, target_size=_target_size)


if LOAD_IMAGES_IN_MEMORY:
    X_img = np.array(p_map(_load_mag, range(len(ml_sample)), desc="Loading magnetograms"))
    X = X_img.reshape(len(X_img), -1).astype(np.float32)
    X_cnn = X_img[:, None, :, :].astype(np.float32)
    print(f"MLP array: {X.shape}  ({X.nbytes / 1e6:.1f} MB)")
    print(f"CNN array: {X_cnn.shape}  ({X_cnn.nbytes / 1e6:.1f} MB)")
else:
    X = None
    X_cnn = None

### MLP classifier

First we try a small **Multi-Layer Perceptron** on flattened magnetogram pixels.

#### Architecture — `SolarMLP`

```
Input         TARGET_SIZE × TARGET_SIZE pixels
Layer 1       Linear (TARGET_SIZE²)→512 → LeakyReLU → Dropout(0.3)
Layer 2       Linear 512→128 → LeakyReLU → Dropout(0.2)
Head          Linear 128→3
```

#### Data loading

The neural-network sections use the `LOAD_IMAGES_IN_MEMORY` switch from the split cell:

* `True` — preload all magnetograms once into `X` / `X_cnn` and train with `TensorDataset`.
* `False` — use `SolarFITSDataset`, which reads and preprocesses FITS files on demand.

The toy dataset is small enough that preloading is feasible and faster.

In [ ]:
# -- MLP setup ----------------------------------------------------------------
if LOAD_IMAGES_IN_MEMORY:
    mlp_train_ds, mlp_val_ds, mlp_test_ds = ut.split_tensor_datasets(
        X, train_idx, val_idx, test_idx, y
    )
else:
    mlp_train_ds = ut.SolarFITSDataset(ml_sample, train_idx, le, FITS_DIR, target_size=TARGET_SIZE, flatten=True)
    mlp_val_ds = ut.SolarFITSDataset(ml_sample, val_idx, le, FITS_DIR, target_size=TARGET_SIZE, flatten=True)
    mlp_test_ds = ut.SolarFITSDataset(ml_sample, test_idx, le, FITS_DIR, target_size=TARGET_SIZE, flatten=True)

mlp_train_dl, mlp_val_dl, mlp_test_dl = ut.make_loaders(
    mlp_train_ds, mlp_val_ds, mlp_test_ds, num_workers=DATALOADER_NUM_WORKERS
)
print(f"MLP data source: {'memory' if LOAD_IMAGES_IN_MEMORY else 'FITS on-the-fly'}")


class SolarMLP(nn.Module):
    def __init__(self, input_dim: int, n_classes: int = 3, negative_slope: float = 0.01):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.LeakyReLU(negative_slope=negative_slope), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.LeakyReLU(negative_slope=negative_slope), nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


mlp_model = SolarMLP(input_dim=TARGET_SIZE * TARGET_SIZE, n_classes=len(le.classes_)).to(DEVICE)
mlp_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {mlp_params:,}")
print(mlp_model)


In [ ]:
# -- MLP training and evaluation ---------------------------------------------
mlp_history, mlp_acc, mlp_true, mlp_preds = ut.fit_evaluate_classifier(
    mlp_model,
    mlp_train_dl,
    mlp_val_dl,
    mlp_test_dl,
    result_key="mlp",
    display_name="MLP",
    model_results=MODEL_RESULTS,
    label_encoder=le,
    lr=MLP_LR,
    epochs=MLP_EPOCHS,
    class_weight=class_weight,
    device=DEVICE,
    print_every=1,
    show_progress=SHOW_PROGRESS_BARS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_metric=EARLY_STOPPING_METRIC,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    restore_best=RESTORE_BEST_WEIGHTS,
    tensorboard_log_dir=TB_LOG_DIR if ENABLE_TENSORBOARD else None,
    run_name="mlp",
    hparams={**TB_COMMON_HPARAMS, "input": "flattened_magnetogram"},
    cmap="Purples",
)


### CNN classifier

We now replace dense-only learning with a small **Convolutional Neural Network** trained end-to-end on the magnetogram image.

CNNs learn spatial filters at multiple scales, detecting features like magnetic dipole orientation, polarity separation and flux complexity that dense layers must infer from many independent pixel weights.

We first train the CNN with a plain, unweighted cross-entropy loss. 

#### Architecture — `SolarCNN`

```
Input         1 × TARGET_SIZE × TARGET_SIZE
Block 1       Conv 3×3 / 32  → BN → LeakyReLU → MaxPool ÷2
Block 2       Conv 3×3 / 64  → BN → LeakyReLU → MaxPool ÷2
Block 3       Conv 3×3 / 128 → BN → LeakyReLU → AdaptiveAvgPool to 4 × 4
Head          Flatten → Linear 2 048→256 → LeakyReLU → Dropout(0.4) → Linear 256→3
```

- **Optimiser:** Adam
- **LR schedule:** cosine annealing over 40 epochs
- **Loss:** plain cross-entropy for the first CNN baseline

#### Data loading

The CNN uses the same `LOAD_IMAGES_IN_MEMORY` switch as the MLP:

* `True` — reshape preloaded `X` to `(N, 1, TARGET_SIZE, TARGET_SIZE)` and train from memory.
* `False` — let `SolarFITSDataset` read and preprocess FITS files on demand.

The model definition is unchanged; only the data source changes.

In [ ]:
# -- CNN setup ----------------------------------------------------------------
if LOAD_IMAGES_IN_MEMORY:
    train_ds, val_ds, test_ds = ut.split_tensor_datasets(X_cnn, train_idx, val_idx, test_idx, y)
else:
    train_ds = ut.SolarFITSDataset(ml_sample, train_idx, le, FITS_DIR, target_size=TARGET_SIZE, flatten=False)
    val_ds = ut.SolarFITSDataset(ml_sample, val_idx, le, FITS_DIR, target_size=TARGET_SIZE, flatten=False)
    test_ds = ut.SolarFITSDataset(ml_sample, test_idx, le, FITS_DIR, target_size=TARGET_SIZE, flatten=False)

train_dl, val_dl, test_dl = ut.make_loaders(
    train_ds, val_ds, test_ds, num_workers=DATALOADER_NUM_WORKERS
)
print(f"CNN data source: {'memory' if LOAD_IMAGES_IN_MEMORY else 'FITS on-the-fly'}")


class SolarCNN(nn.Module):
    def __init__(self, n_classes: int = 3, negative_slope: float = 0.01):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(negative_slope=negative_slope),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(negative_slope=negative_slope),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(negative_slope=negative_slope),
            nn.AdaptiveAvgPool2d(4),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.LeakyReLU(negative_slope=negative_slope), nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


cnn_uw_model = SolarCNN(n_classes=len(le.classes_)).to(DEVICE)
n_params = sum(p.numel() for p in cnn_uw_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(cnn_uw_model)


In [ ]:
# -- Unweighted CNN training and evaluation -----------------------------------
cnn_uw_history, cnn_uw_acc, uw_true, uw_preds = ut.fit_evaluate_classifier(
    cnn_uw_model,
    train_dl,
    val_dl,
    test_dl,
    result_key="cnn_unweighted",
    display_name="CNN unweighted",
    model_results=MODEL_RESULTS,
    label_encoder=le,
    lr=LR,
    epochs=N_EPOCHS,
    class_weight=None,
    device=DEVICE,
    print_every=1,
    show_progress=SHOW_PROGRESS_BARS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_metric=EARLY_STOPPING_METRIC,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    restore_best=RESTORE_BEST_WEIGHTS,
    tensorboard_log_dir=TB_LOG_DIR if ENABLE_TENSORBOARD else None,
    run_name="cnn_unweighted",
    hparams={**TB_COMMON_HPARAMS, "input": "magnetogram", "class_weighted": False},
    baselines={"MLP": MODEL_RESULTS["mlp"]["accuracy"]} if "mlp" in MODEL_RESULTS else None,
    cmap="Oranges",
)
if "mlp" in MODEL_RESULTS:
    print(f"Gain over MLP: {cnn_uw_acc - MODEL_RESULTS['mlp']['accuracy']:+.3f}")


### Class-weighting

The unweighted CNN can optimise majority-class accuracy by favouring QS. We now train the same architecture again with **balanced class weights** so that AR, IA and QS contribute equally to the loss.

This just changes the training objective.

In [ ]:
# -- Weighted CNN: same architecture, balanced CrossEntropyLoss ---------------
cnn_model = SolarCNN(n_classes=len(le.classes_)).to(DEVICE)
n_w_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_w_params:,}")

_baselines = {}
if "cnn_unweighted" in MODEL_RESULTS:
    _baselines["Unweighted CNN"] = MODEL_RESULTS["cnn_unweighted"]["accuracy"]
if "mlp" in MODEL_RESULTS:
    _baselines["MLP"] = MODEL_RESULTS["mlp"]["accuracy"]

cnn_history, cnn_acc, cnn_true, cnn_preds = ut.fit_evaluate_classifier(
    cnn_model,
    train_dl,
    val_dl,
    test_dl,
    result_key="cnn_weighted",
    display_name="CNN weighted",
    model_results=MODEL_RESULTS,
    label_encoder=le,
    lr=LR,
    epochs=N_EPOCHS,
    class_weight=class_weight,
    device=DEVICE,
    print_every=1,
    show_progress=SHOW_PROGRESS_BARS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_metric=EARLY_STOPPING_METRIC,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    restore_best=RESTORE_BEST_WEIGHTS,
    tensorboard_log_dir=TB_LOG_DIR if ENABLE_TENSORBOARD else None,
    run_name="cnn_weighted",
    hparams={**TB_COMMON_HPARAMS, "input": "magnetogram", "class_weighted": True},
    baselines=_baselines,
    cmap="Oranges",
)
if "cnn_unweighted" in MODEL_RESULTS:
    print(f"Gain over unweighted CNN: {cnn_acc - MODEL_RESULTS['cnn_unweighted']['accuracy']:+.3f}")


### 2D CNN classifier with early fusion (magnetogram + continuum)

For the multimodal baseline, we use the simpler production-style approach: stack magnetogram and continuum as two ordinary image channels and use a standard 2D CNN.

Input shape is `2 × TARGET_SIZE × TARGET_SIZE`: channel 0 is the magnetogram, channel 1 is the continuum. The first `Conv2d` layer performs the fusion by learning filters across both channels.


In [ ]:
# -- Load or stream mag + cont as a 2D early-fusion tensor ---------------------
# Channel 0: magnetogram -> preprocess_image (divide by 800 G, hardtanh [-1, 1])
# Channel 1: continuum   -> preprocess_continuum (inverted per-image min-max [0, 1])
print("Preparing mag+cont pairs as a 2D early-fusion tensor ...")


def _load_dual(idx, _target_size=TARGET_SIZE):
    """Load, preprocess, and stack mag + cont as (C=2, H, W)."""
    row = ml_sample.iloc[idx]
    mag, cont = ut.load_fits_pair(row, FITS_DIR)
    mag_p = ut.preprocess_image(mag, target_size=_target_size)
    cont_p = ut.preprocess_continuum(cont, target_size=_target_size)
    return np.stack([mag_p, cont_p], axis=0).astype(np.float32)


if LOAD_IMAGES_IN_MEMORY:
    X_dual = np.array(p_map(_load_dual, range(len(ml_sample)), desc="Loading dual-channel images"))
    print(f"2D fusion CNN array: {X_dual.shape}  ({X_dual.nbytes / 1e6:.1f} MB)")
    train_ds_dual, val_ds_dual, test_ds_dual = ut.split_tensor_datasets(
        X_dual, train_idx, val_idx, test_idx, y
    )
else:
    X_dual = None
    train_ds_dual = ut.SolarFITSDualDataset(
        ml_sample, train_idx, le, FITS_DIR, target_size=TARGET_SIZE, as_volume=False
    )
    val_ds_dual = ut.SolarFITSDualDataset(
        ml_sample, val_idx, le, FITS_DIR, target_size=TARGET_SIZE, as_volume=False
    )
    test_ds_dual = ut.SolarFITSDualDataset(
        ml_sample, test_idx, le, FITS_DIR, target_size=TARGET_SIZE, as_volume=False
    )

train_dl_dual, val_dl_dual, test_dl_dual = ut.make_loaders(
    train_ds_dual, val_ds_dual, test_ds_dual, num_workers=DATALOADER_NUM_WORKERS
)
print(f"2D fusion CNN data source: {'memory' if LOAD_IMAGES_IN_MEMORY else 'FITS on-the-fly'}")


#### Architecture — `SolarDualCNN`

```
Input         2 × TARGET_SIZE × TARGET_SIZE
Block 1       Conv 3×3 / 32  → BN → LeakyReLU → MaxPool ÷2
Block 2       Conv 3×3 / 64  → BN → LeakyReLU → MaxPool ÷2
Block 3       Conv 3×3 / 128 → BN → LeakyReLU → AdaptiveAvgPool to 4 × 4
Head          Flatten → Linear 2 048→256 → LeakyReLU → Dropout(0.4) → Linear 256→3
```

This is basic early fusion: the architecture is almost identical to `SolarCNN`, except the first convolution receives two input channels instead of one.


In [ ]:
# -- 2D early-fusion CNN -------------------------------------------------------
class SolarDualCNN(nn.Module):
    """2D CNN over magnetogram and continuum as two image channels."""

    def __init__(self, n_classes: int = 3, negative_slope: float = 0.01):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 32, 3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(negative_slope=negative_slope),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(negative_slope=negative_slope),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(negative_slope=negative_slope),
            nn.AdaptiveAvgPool2d(4),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.LeakyReLU(negative_slope=negative_slope), nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


cnn_2d_fusion_model = SolarDualCNN(n_classes=len(le.classes_)).to(DEVICE)
cnn_2d_fusion_params = sum(p.numel() for p in cnn_2d_fusion_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {cnn_2d_fusion_params:,}")
print(cnn_2d_fusion_model)

_baselines = {}
if "cnn_weighted" in MODEL_RESULTS:
    _baselines["CNN weighted"] = MODEL_RESULTS["cnn_weighted"]["accuracy"]
if "cnn_unweighted" in MODEL_RESULTS:
    _baselines["CNN unweighted"] = MODEL_RESULTS["cnn_unweighted"]["accuracy"]

cnn_2d_fusion_history, cnn_2d_fusion_acc, cnn_2d_fusion_true, cnn_2d_fusion_preds = ut.fit_evaluate_classifier(
    cnn_2d_fusion_model,
    train_dl_dual,
    val_dl_dual,
    test_dl_dual,
    result_key="cnn_2d_fusion",
    display_name="CNN 2D mag+cont",
    model_results=MODEL_RESULTS,
    label_encoder=le,
    lr=LR,
    epochs=N_EPOCHS,
    class_weight=class_weight,
    device=DEVICE,
    print_every=1,
    show_progress=SHOW_PROGRESS_BARS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_metric=EARLY_STOPPING_METRIC,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    restore_best=RESTORE_BEST_WEIGHTS,
    tensorboard_log_dir=TB_LOG_DIR if ENABLE_TENSORBOARD else None,
    run_name="cnn_2d_mag_cont",
    hparams={**TB_COMMON_HPARAMS, "input": "magnetogram_plus_continuum", "convolution": "2d", "fusion": "early_channels"},
    baselines=_baselines,
    cmap="Greens",
)
if "cnn_weighted" in MODEL_RESULTS:
    print(f"Gain over 1ch CNN: {cnn_2d_fusion_acc - MODEL_RESULTS['cnn_weighted']['accuracy']:+.3f}")


In [ ]:
# ── Final model comparison ---------------------------------------------------
comparison_df = ut.metrics_table(MODEL_RESULTS, le)
display(comparison_df.round(3))


In [ ]:
# ── Confusion matrix comparison ---------------------------------------------
# Side-by-side row-normalised confusion matrices with counts and percentages.
ut.plot_confusion_matrix_grid(MODEL_RESULTS, le, n_cols=2)


In [ ]:
ut.show_tensorboard(TB_LOG_DIR)

## Ideas to mess around with
Most of these can be tried by changing only the two configuration cells and rerunning the affected sections. TensorBoard is useful here because each model call logs a separate run.

### Config experiments
- Change `TARGET_SIZE`: try `64`, `128`, and `200`. Watch whether higher resolution helps AR recall enough to justify the slower training.
- Change `N_EPOCHS` and `MLP_EPOCHS`: compare short runs with more stable runs (`30`-`100` epochs).
- Toggle `LOAD_IMAGES_IN_MEMORY`: use `True` for speed on the toy dataset; use `False` to mimic production-style streaming.
- Change `EARLY_STOPPING_METRIC`: compare `"val_loss"` with `"val_macro_f1"` and check whether the selected checkpoint shifts toward better minority-class recall.
- Change `EARLY_STOPPING_PATIENCE`: try `None`, `4`, and `12` to see how much overfitting is hidden by best-weight restoration.
- Change `TEST_FOLDS`, `VAL_FOLDS`, and `TRAIN_FOLDS`: keep them disjoint, then check whether model ranking is stable across split choices.

### Data and preprocessing questions
- Train with only HMI or only MDI rows.
- Try continuum-only classification by changing the loading function to use `ut.preprocess_continuum` as the single input.

### Model experiments
- Reduce the CNN width, for example `16/32/64` instead of `32/64/128`, and check whether performance changes much.
- Increase dropout in the MLP/CNN heads and see whether validation curves become less noisy.
- Freeze the split and run the same model several times with different random seeds to estimate run-to-run variance.

### Evaluation ideas
- Break down confusion matrices by instrument (`MDI` vs `HMI`) to check whether one instrument is driving errors.
- Plot accuracy or macro-F1 against observation year to look for temporal drift.
